# Evaluation of IDF of hourly CPM emulators

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.furflex_default_params import *

In [ ]:
import dask
import dask.array
from dask.distributed import Client
import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from mlde_analysis.idf import plot_pmfs, NORM, DIFF_NORM, REL_DIFF_NORM, calc_pmf_ndimage

In [ ]:
client = Client()
client

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.furflex_magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS[target_sim_key]

## IDF

In [ ]:
%%time

var = "pr"
pmfs = {}

target_pmf = TARGET_DAS[var].groupby(["time.season", "time.year"]).map(calc_pmf_ndimage).mean(dim=["season", "year"])

pred_pmf = PRED_DAS[var].groupby(["time.season", "time.year"]).map(calc_pmf_ndimage).mean(dim=["season", "year"]).mean("sample_id")

### Probabilities

In [ ]:
%%time

plot_pmfs(target_pmf, pred_pmf, norm=NORM)
plt.show()

### Error

In [ ]:
plot_pmfs(target_pmf, pred_pmf - target_pmf, target_cbar=True, norm=DIFF_NORM, cmap="RdBu", cbar_label="Frequency error")
plt.show()

### Relative error

In [ ]:
plot_pmfs(target_pmf, 100* (pred_pmf - target_pmf)/target_pmf, target_cbar=True, norm=REL_DIFF_NORM, cmap="RdBu", cbar_label="Relative frequency error (%)")
plt.show()

## Seasonal IDF

In [ ]:
%%time
var = "pr"
pmfs = {}

seasonal_target_pmf = TARGET_DAS[var].groupby(["time.season", "time.year"]).map(calc_pmf_ndimage).mean(dim=["year"])

seasonal_pred_pmf = PRED_DAS[var].groupby(["time.season", "time.year"]).map(calc_pmf_ndimage).mean(dim=["year"]).mean("sample_id")

In [ ]:
for season, season_target_pmf in seasonal_target_pmf.groupby("season"):
    IPython.display.display_markdown(f"### {season}", raw=True)
    season_pred_pmf = seasonal_pred_pmf.sel(season=season)
    plot_pmfs(season_target_pmf.squeeze("season"), season_pred_pmf, norm=NORM)
    plt.show()

In [ ]:
client.close()